# Stage 3 — Sentiment Analysis (NLP Models)

Fine-tunes each NLP model on weak-labeled AMZN headlines with a
chronological split, logs metrics/timings, saves `./results/nlp_results.csv`,
and writes best-model headline probabilities to `./artifacts/nlp_probs.parquet`.

> **Energy hook:** training and inference remain bracketed by
> `time.time()` so energy instrumentation can be attached at the same points.


In [1]:
# Load shared helpers, config values, and model registries.
from common import *

# Hugging Face training utilities used in this stage.
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
)

# text_df contains weak-labeled AMZN headlines from Stage 1.
text_df = load_text_df()
# num_df is loaded only to compute a shared split cutoff date.
num_df = load_num_df()

# Normalize timestamps to date-level precision for clean chronological splits.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# cutoff_date defines the train/test boundary used by both text and numeric tasks.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
print(f"Chronological cutoff: {cutoff_date.date()}")

# Collector object accumulates per-run metrics for CSV export.
results = ResultsCollector()

[common] device=cuda  artifacts=/cluster/tufts/c26sp1cee0132/jmonta04/ai_project/module_version/artifacts  results=/cluster/tufts/c26sp1cee0132/jmonta04/ai_project/module_version/results


FileNotFoundError: [Errno 2] No such file or directory: '/cluster/tufts/c26sp1cee0132/jmonta04/ai_project/module_version/artifacts/text_df.parquet'

## 3.1 Training routine

In [ ]:
def build_tokenizer_and_model(model_path, config):
    """Create tokenizer/model pair for a checkpoint path."""
    # GPT-2 requires special handling for padding tokens.
    is_gpt2 = "gpt2" in model_path
    if is_gpt2:
        tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token

        model = GPT2ForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        # Non-GPT models can use Auto classes directly.
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )

    # Move model to GPU/CPU selected in common.py.
    model.to(DEVICE)
    return tokenizer, model


def train_nlp_model(model_name, model_path, text_df, config, seed, cutoff_date):
    """Train one model for one seed and return metrics + predictions."""
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=config)

    # Build chronological train/test datasets to avoid temporal leakage.
    train_ds, test_ds, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=config["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    # run_dir stores temporary trainer outputs for this run.
    run_dir = RESULTS_DIR / f"{model_name.replace(' ', '_')}_seed{seed}"
    training_args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["train_batch_size"],
        per_device_eval_batch_size=config["eval_batch_size"],
        learning_rate=config["learning_rate"],
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        seed=seed,
        report_to="none",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=config.get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_clf_metrics,
    )

    # Measure end-to-end training time for this run.
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    # Evaluate on chronological test split.
    eval_out = trainer.evaluate()

    # Measure prediction/inference time separately.
    t0 = time.time()
    pred_out = trainer.predict(test_ds)
    infer_time = time.time() - t0

    # Convert logits to class IDs via argmax.
    preds = np.argmax(pred_out.predictions, axis=-1)

    # record stores metrics that will be appended to nlp_results.csv.
    record = {
        "model": model_name,
        "seed": seed,
        "accuracy": eval_out["eval_accuracy"],
        "f1": eval_out["eval_f1"],
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(test_ds),
    }

    # Free memory before the next model/seed run.
    del model, trainer
    torch.cuda.empty_cache()

    return record, preds


def train_best_model_and_export_probs(model_name, model_path, seed, text_df, cutoff_date):
    """Retrain best run and export per-headline class probabilities."""
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=CONFIG["nlp"])

    # Fit using chronological training slice only.
    train_ds, _, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    output_dir = RESULTS_DIR / f"best_{model_name.replace(' ', '_')}_seed{seed}"
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=CONFIG["nlp"]["epochs"],
        per_device_train_batch_size=CONFIG["nlp"]["train_batch_size"],
        per_device_eval_batch_size=CONFIG["nlp"]["eval_batch_size"],
        learning_rate=CONFIG["nlp"]["learning_rate"],
        save_strategy="no",
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=CONFIG["nlp"].get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
    )
    trainer.train()

    # Create inference dataset for every headline row.
    # Labels are placeholders because we only need probabilities here.
    infer_ds = SentimentDataset(
        texts=text_df["text"],
        labels=pd.Series(np.zeros(len(text_df), dtype=int)),
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
    )

    prediction_output = trainer.predict(infer_ds)
    logits = prediction_output.predictions
    # Softmax converts logits into class probabilities summing to 1.
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    # Save one row per headline so Stage 4 can build rolling sentiment features.
    nlp_probs_df = pd.DataFrame(
        {
            "date": text_df["date"].values,
            "prob_negative": probs[:, LABEL_TO_ID["negative"]],
            "prob_neutral": probs[:, LABEL_TO_ID["neutral"]],
            "prob_positive": probs[:, LABEL_TO_ID["positive"]],
            "model": model_name,
            "seed": seed,
        }
    )

    # Free memory after export.
    del model, trainer
    torch.cuda.empty_cache()

    return nlp_probs_df

## 3.2 Run every (model, seed) combination

In [ ]:
# nlp_predictions keeps raw predicted class IDs for optional diagnostics.
nlp_predictions = {}

# Train every model across every configured seed.
for model_name, model_path in NLP_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        print(f"\n{'=' * 60}")
        print(f"  {model_name}  |  seed {seed}")
        print(f"{'=' * 60}")

        # Train and evaluate one (model, seed) run.
        record, preds = train_nlp_model(
            model_name=model_name,
            model_path=model_path,
            text_df=text_df,
            config=CONFIG["nlp"],
            seed=seed,
            cutoff_date=cutoff_date,
        )

        # Save metrics and predictions for this run.
        results.add_nlp(record)
        nlp_predictions[(model_name, seed)] = preds

        # Print compact run summary.
        print(f"  Accuracy: {record['accuracy']:.4f}  |  F1: {record['f1']:.4f}")
        print(
            f"  Train: {record['train_time_s']:.1f}s  |  "
            f"Infer: {record['infer_time_s']:.1f}s"
        )

# Preview collected NLP metrics.
results.nlp_df().round(6)

## 3.3 Persist results and export best-model headline probabilities

In [ ]:
# Save Stage 3 metrics to results/nlp_results.csv.
results.save(RESULTS_DIR)

# Identify the best run by F1 so Stage 4 can consume a single probability file.
nlp_results_df = results.nlp_df().copy()
best_row = nlp_results_df.sort_values("f1", ascending=False).iloc[0]

# Extract model ID and seed of the top run.
best_model_name = best_row["model"]
best_seed = int(best_row["seed"])
best_model_path = NLP_MODELS[best_model_name]

print("Best NLP run used for probability export:")
print(best_row.to_string())

# Re-train that best configuration and export per-headline probabilities.
nlp_probs_df = train_best_model_and_export_probs(
    model_name=best_model_name,
    model_path=best_model_path,
    seed=best_seed,
    text_df=text_df,
    cutoff_date=cutoff_date,
)
save_nlp_probs_df(nlp_probs_df)

print(f"Total NLP experiments: {len(results.nlp_results)}")
print(f"Saved per-headline probabilities to {ARTIFACTS_DIR / 'nlp_probs.parquet'}")